<a href="https://colab.research.google.com/github/PravarGarg/CropDetection_DeepLearning/blob/main/Experiment2_Salinas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hybrid ViT + Graph Transformer for Hyperspectral Crop Classification & Label-Efficient Graph Learning
This notebook implements:
1. **Preprocessing & Superpixel Graph Construction:** PCA dimensionality reduction (`C=30`), SLIC superpixel segmentation, and symmetric $k$-NN adjacency matrix normalization.
2. **Hybrid ViT + Graph Transformer Architecture:** Dual-branch pixel-level ViT encoder and region-level GCN + Graph Transformer encoder.
3. **Experiment 1 (Full Supervised Baseline):** Standard 70/30 train-test evaluation.
4. **Experiment 2 (Label-Efficient Graph Learning):**
   - **Stage 1:** Self-supervised masked pretraining of both branches on all scene pixels (no labels).
   - **Stage 2:** Low-label fine-tuning (`1%`, `5%`, `10%` labels) with intra-superpixel consistency loss and graph-guided pseudo-label propagation.
   - **Benchmark & Plotting:** Comparison against Plain ViT and Supervised ViT+Graph across label fractions.

In [ ]:
# Cell 1: Imports & Hyperparameters
import os
import copy
import urllib.request
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from skimage.segmentation import slic

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Model & Data Hyperparameters
PCA_COMPONENTS = 30
PATCH_SIZE = 7
N_SEGMENTS = 200
COMPACTNESS = 0.1
KNN_K = 5
BATCH_SIZE = 128
EPOCHS = 15
LR = 1e-3

Using device: cpu


In [ ]:
# Cell 2: Download & Load Salinas Hyperspectral Dataset
def load_salinas_dataset(data_dir='./data'):
    os.makedirs(data_dir, exist_ok=True)
    img_path = os.path.join(data_dir, 'Salinas_corrected.mat')
    gt_path = os.path.join(data_dir, 'Salinas_gt.mat')

    files_and_mirrors = {
        img_path: [
            'https://raw.githubusercontent.com/gokriznastic/HybridSN/master/data/Salinas_corrected.mat',
            'https://www.ehu.eus/ccwintco/uploads/a/a3/Salinas_corrected.mat',
        ],
        gt_path: [
            'https://raw.githubusercontent.com/gokriznastic/HybridSN/master/data/Salinas_gt.mat',
            'https://www.ehu.eus/ccwintco/uploads/f/fa/Salinas_gt.mat',
        ]
    }

    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

    for path, urls in files_and_mirrors.items():
        if not os.path.exists(path):
            downloaded = False
            for url in urls:
                try:
                    print(f'Downloading {os.path.basename(path)} from {url}...')
                    req = urllib.request.Request(url, headers=headers)
                    with urllib.request.urlopen(req, timeout=30) as resp, open(path, 'wb') as out_file:
                        out_file.write(resp.read())
                    downloaded = True
                    break
                except Exception as e:
                    print(f'Mirror failed ({e}), trying next...')
            if not downloaded:
                raise RuntimeError(f'Could not download {os.path.basename(path)} from any mirror.')

    X = sio.loadmat(img_path)['salinas_corrected'].astype(np.float32)
    y = sio.loadmat(gt_path)['salinas_gt'].astype(np.int64)
    return X, y

X_raw, y = load_salinas_dataset()
H, W, B = X_raw.shape
num_classes = int(y.max())
print(f'Salinas Cube Shape: {X_raw.shape} | Classes: {num_classes} | Labelled Pixels: {(y > 0).sum()}')

Salinas Cube Shape: (512, 217, 204) | Classes: 16 | Labelled Pixels: 54129


In [ ]:
# Cell 3: Spectral Standardization & PCA Dimensionality Reduction
class PCAReducer:
    def __init__(self, n_components=30):
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_components, random_state=SEED)

    def fit_transform(self, X):
        h, w, b = X.shape
        X_flat = X.reshape(-1, b)
        X_scaled = self.scaler.fit_transform(X_flat)
        X_pca = self.pca.fit_transform(X_scaled)
        return X_pca.reshape(h, w, -1).astype(np.float32)

reducer = PCAReducer(n_components=PCA_COMPONENTS)
X_pca = reducer.fit_transform(X_raw)
print(f'Reduced HSI Shape: {X_pca.shape} | Explained Variance: {reducer.pca.explained_variance_ratio_.sum()*100:.2f}%')

Reduced HSI Shape: (512, 217, 30) | Explained Variance: 99.96%


In [ ]:
# Cell 4: SLIC Superpixel Segmentation & Graph Construction
def build_superpixel_graph(X_pca, n_segments=200, compactness=0.1, k=5):
    # Normalize first 3 PCs to [0, 1] for SLIC
    rgb = X_pca[:, :, :3].copy()
    for i in range(3):
        c_min, c_max = rgb[:, :, i].min(), rgb[:, :, i].max()
        rgb[:, :, i] = (rgb[:, :, i] - c_min) / (c_max - c_min + 1e-8)

    segments = slic(rgb, n_segments=n_segments, compactness=compactness, start_label=0)
    # Remap segment IDs contiguously to [0, N-1]
    unique_segs, inverse = np.unique(segments, return_inverse=True)
    segments = inverse.reshape(X_pca.shape[:2])
    num_nodes = len(unique_segs)

    # Node features: average 30-d PCA feature per superpixel
    C = X_pca.shape[2]
    node_features = np.zeros((num_nodes, C), dtype=np.float32)
    flat_X = X_pca.reshape(-1, C)
    flat_segs = segments.reshape(-1)
    np.add.at(node_features, flat_segs, flat_X)
    counts = np.bincount(flat_segs, minlength=num_nodes)[:, None].astype(np.float32)
    node_features /= np.maximum(counts, 1.0)

    # Symmetric k-NN adjacency matrix with self-loops: D^(-1/2) A D^(-1/2)
    knn_adj = kneighbors_graph(node_features, n_neighbors=min(k, num_nodes - 1), mode='connectivity', include_self=False)
    adj = knn_adj.toarray().astype(np.float32)
    adj = np.maximum(adj, adj.T) + np.eye(num_nodes, dtype=np.float32)

    deg = np.sum(adj, axis=1)
    deg_inv_sqrt = np.power(deg, -0.5)
    deg_inv_sqrt[np.isinf(deg_inv_sqrt)] = 0.0
    D_inv_sqrt = np.diag(deg_inv_sqrt)
    adj_norm = (D_inv_sqrt @ adj @ D_inv_sqrt).astype(np.float32)

    return segments, node_features, adj_norm

segments, node_features, adj_norm = build_superpixel_graph(
    X_pca, n_segments=N_SEGMENTS, compactness=COMPACTNESS, k=KNN_K
)
print(f'Superpixel Graph Built | Nodes: {node_features.shape[0]} | Node Feature Dim: {node_features.shape[1]}')

Superpixel Graph Built | Nodes: 39 | Node Feature Dim: 30


In [ ]:
# Cell 5: Patch Dataset & Train/Test Split for Supervised Baseline
class PatchDataset(Dataset):
    def __init__(self, X_pca, y, segments, coords, patch_size=7):
        self.pad = patch_size // 2
        self.X_pad = np.pad(X_pca, ((self.pad, self.pad), (self.pad, self.pad), (0, 0)), mode='reflect')
        self.y = y
        self.segments = segments
        self.coords = coords

    def __len__(self):return len(self.coords)

    def __getitem__(self, idx):
        r, c = self.coords[idx]
        patch = self.X_pad[r:r + 2*self.pad + 1, c:c + 2*self.pad + 1, :]
        patch = torch.tensor(patch, dtype=torch.float32).permute(2, 0, 1)  # (C, 7, 7)
        label = torch.tensor(self.y[r, c] - 1, dtype=torch.long)
        seg_id = torch.tensor(self.segments[r, c], dtype=torch.long)
        return patch, label, seg_id

coords = [(r, c) for r in range(H) for c in range(W) if y[r, c] > 0]
targets = [y[r, c] - 1 for r, c in coords]

train_coords, test_coords = train_test_split(
    coords, test_size=0.30, stratify=targets, random_state=SEED
)

train_ds = PatchDataset(X_pca, y, segments, train_coords, patch_size=PATCH_SIZE)
test_ds = PatchDataset(X_pca, y, segments, test_coords, patch_size=PATCH_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
print(f'70/30 Split -> Train: {len(train_ds)} | Test: {len(test_ds)}')

70/30 Split -> Train: 37890 | Test: 16239


In [ ]:
# Cell 6: Hybrid ViT + Graph Transformer Model Architecture
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=30, embed_dim=128, patch_size=7):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=3, padding=1)
        self.pos_embed = nn.Parameter(torch.zeros(1, patch_size * patch_size, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        # x: (B, C, 7, 7) -> (B, 49, embed_dim)
        x = self.proj(x).flatten(2).transpose(1, 2)
        return x + self.pos_embed


class GCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x, adj_norm):
        return torch.matmul(adj_norm, self.linear(x))


class GraphTransformer(nn.Module):
    def __init__(self, dim=64, heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim)
        )
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        # x: (N_nodes, dim)
        h = x.unsqueeze(0)
        attn_out, _ = self.attn(h, h, h)
        h = self.norm1(h + attn_out)
        h = self.norm2(h + self.ffn(h))
        return h.squeeze(0)


class ViTGraphNet(nn.Module):
    def __init__(self, in_channels=30, num_classes=16, vit_dim=128, graph_dim=64):
        super().__init__()
        # Branch 1: Pixel-Level Local ViT
        self.vit_embed = PatchEmbedding(in_channels, vit_dim, patch_size=PATCH_SIZE)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=vit_dim, nhead=4, dim_feedforward=256,
            dropout=0.1, activation='gelu', batch_first=True, norm_first=True
        )
        self.vit_encoder = nn.TransformerEncoder(enc_layer, num_layers=2)

        # Branch 2: Region-Level Global Graph (GCN + Graph Transformer)
        self.gcn1 = GCNLayer(in_channels, graph_dim)
        self.gcn2 = GCNLayer(graph_dim, graph_dim)
        self.graph_tf = GraphTransformer(dim=graph_dim, heads=4)

        # Fusion & Classifier
        self.classifier = nn.Sequential(
            nn.Linear(vit_dim + graph_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, patches, seg_ids, node_feats, adj_norm):
        # ViT branch
        v = self.vit_embed(patches)
        v = self.vit_encoder(v).mean(dim=1)  # (B, 128)

        # Graph branch
        g = F.relu(self.gcn1(node_feats, adj_norm))
        g = F.relu(self.gcn2(g, adj_norm))
        g = self.graph_tf(g)  # (N_nodes, 64)
        g_pixel = g[seg_ids]  # (B, 64)

        # Concatenate & Classify
        fused = torch.cat([v, g_pixel], dim=-1)  # (B, 192)
        return self.classifier(fused)

In [ ]:
# Cell 7: Evaluation Utility & Experiment 1 (70% Labels Baseline)
def evaluate(model, loader, node_t, adj_t):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for patches, labels, seg_ids in loader:
            patches, seg_ids = patches.to(device), seg_ids.to(device)
            logits = model(patches, seg_ids, node_t, adj_t)
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    oa = accuracy_score(all_labels, all_preds)
    cm = confusion_matrix(all_labels, all_preds)
    per_class_acc = np.diag(cm) / np.maximum(cm.sum(axis=1), 1)
    aa = np.mean(per_class_acc)
    kappa = cohen_kappa_score(all_labels, all_preds)
    return oa, aa, kappa

node_t = torch.tensor(node_features, dtype=torch.float32, device=device)
adj_t = torch.tensor(adj_norm, dtype=torch.float32, device=device)

model_exp1 = ViTGraphNet(in_channels=PCA_COMPONENTS, num_classes=num_classes).to(device)
optimizer = torch.optim.AdamW(model_exp1.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

print('--- Running Experiment 1: 70% Supervised Training ---')
for epoch in range(1, EPOCHS + 1):
    model_exp1.train()
    total_loss = 0.0
    for patches, labels, seg_ids in train_loader:
        patches, labels, seg_ids = patches.to(device), labels.to(device), seg_ids.to(device)
        optimizer.zero_grad()
        loss = criterion(model_exp1(patches, seg_ids, node_t, adj_t), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:02d}/{EPOCHS:02d} | Loss: {total_loss/len(train_loader):.4f}')

oa, aa, kappa = evaluate(model_exp1, test_loader, node_t, adj_t)
print(f'Experiment 1 (70% Labels) -> OA: {oa*100:.2f}% | AA: {aa*100:.2f}% | Kappa: {kappa*100:.2f}%')

/tmp/ipykernel_1128/100117869.py:55: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.vit_encoder = nn.TransformerEncoder(enc_layer, num_layers=2)


--- Running Experiment 1: 70% Supervised Training ---
Epoch 01/15 | Loss: 0.2228
Epoch 05/15 | Loss: 0.0233
Epoch 10/15 | Loss: 0.0077
Epoch 15/15 | Loss: 0.0048
Experiment 1 (70% Labels) -> OA: 99.78% | AA: 99.82% | Kappa: 99.76%


---
## Experiment 2: Label-Efficient Graph Learning (`1%`, `5%`, `10%` Labels)
1. **Self-Supervised Masked Pretraining:** Trains both the ViT branch (masked patch reconstruction) and Graph branch (masked superpixel feature reconstruction) on **all pixels in the scene with zero labels**.
2. **Graph-Guided Semi-Supervised Fine-Tuning:** Trains on `1%`, `5%`, and `10%` of labels using:
   - **Superpixel Consistency Loss:** Pushes unlabelled pixels within the same superpixel to predict consistent probability distributions.
   - **Graph Pseudo-Label Propagation:** Propagates confident superpixel predictions across $\hat{A}$ to neighbouring unlabelled pixels.

In [ ]:
# Cell 8: Self-Supervised Masked Pretraining on All Scene Pixels (No Labels)
class AllPixelsPatchDataset(Dataset):
    """Extracts 7x7 patches for ALL H*W pixels in the scene without using labels."""
    def __init__(self, X_pca, segments, patch_size=7):
        self.pad = patch_size // 2
        self.H, self.W, self.C = X_pca.shape
        self.X_pad = np.pad(X_pca, ((self.pad, self.pad), (self.pad, self.pad), (0, 0)), mode='reflect')
        self.coords = [(r, c) for r in range(self.H) for c in range(self.W)]
        self.segments = segments

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        r, c = self.coords[idx]
        patch = self.X_pad[r:r + 2*self.pad + 1, c:c + 2*self.pad + 1, :]
        patch = torch.tensor(patch, dtype=torch.float32).permute(2, 0, 1)  # (C, 7, 7)
        seg_id = torch.tensor(self.segments[r, c], dtype=torch.long)
        return patch, seg_id


class MaskedSSLPretrainer(nn.Module):
    """Jointly pretrains the ViT branch (Masked Patch Modeling) and Graph branch (Masked Node Modeling)."""
    def __init__(self, base_model, in_channels=30, embed_dim=128, graph_dim=64):
        super().__init__()
        self.vit_embed = base_model.vit_embed
        self.vit_encoder = base_model.vit_encoder
        self.gcn1 = base_model.gcn1
        self.gcn2 = base_model.gcn2
        self.graph_tf = base_model.graph_tf

        self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)

        self.vit_decoder = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, in_channels)
        )
        self.graph_decoder = nn.Sequential(
            nn.Linear(graph_dim, graph_dim),
            nn.GELU(),
            nn.Linear(graph_dim, in_channels)
        )

    def forward(self, patches, node_feats, adj_norm, patch_mask_ratio=0.4, node_mask_ratio=0.3):
        B, C, H, W = patches.shape
        # Branch 1: Masked Patch Token Reconstruction
        tokens = self.vit_embed(patches)  # (B, 49, 128)
        N_tok = tokens.size(1)
        token_mask = (torch.rand(B, N_tok, device=patches.device) < patch_mask_ratio)
        masked_tokens = torch.where(token_mask.unsqueeze(-1), self.mask_token, tokens)
        encoded_tokens = self.vit_encoder(masked_tokens)
        pred_patches = self.vit_decoder(encoded_tokens)  # (B, 49, C)

        target_patches = patches.flatten(2).transpose(1, 2)  # (B, 49, C)
        if token_mask.sum() > 0:
            loss_vit = F.mse_loss(pred_patches[token_mask], target_patches[token_mask])
        else:
            loss_vit = F.mse_loss(pred_patches, target_patches)

        # Branch 2: Masked Superpixel Node Reconstruction
        N_nodes = node_feats.size(0)
        node_mask = (torch.rand(N_nodes, 1, device=node_feats.device) < node_mask_ratio)
        masked_nodes = torch.where(node_mask, torch.zeros_like(node_feats), node_feats)
        g = F.relu(self.gcn1(masked_nodes, adj_norm))
        g = F.relu(self.gcn2(g, adj_norm))
        g = self.graph_tf(g)
        pred_nodes = self.graph_decoder(g)
        if node_mask.sum() > 0:
            loss_graph = F.mse_loss(pred_nodes[node_mask.squeeze(-1)], node_feats[node_mask.squeeze(-1)])
        else:
            loss_graph = F.mse_loss(pred_nodes, node_feats)

        return loss_vit + loss_graph


def pretrain_ssl(X_pca, segments, node_features, adj_norm, num_classes, ssl_epochs=10, batch_size=256, lr=1e-3):
    print(f'--- Starting Self-Supervised Pretraining on all {X_pca.shape[0]*X_pca.shape[1]} pixels ---')
    base_model = ViTGraphNet(in_channels=X_pca.shape[2], num_classes=num_classes).to(device)
    ssl_wrapper = MaskedSSLPretrainer(base_model, in_channels=X_pca.shape[2]).to(device)

    all_ds = AllPixelsPatchDataset(X_pca, segments, patch_size=PATCH_SIZE)
    all_loader = DataLoader(all_ds, batch_size=batch_size, shuffle=True, drop_last=False)

    node_t = torch.tensor(node_features, dtype=torch.float32, device=device)
    adj_t = torch.tensor(adj_norm, dtype=torch.float32, device=device)
    optimizer = torch.optim.AdamW(ssl_wrapper.parameters(), lr=lr, weight_decay=1e-4)

    ssl_wrapper.train()
    for epoch in range(1, ssl_epochs + 1):
        total_loss = 0.0
        for patches, _ in all_loader:
            patches = patches.to(device)
            optimizer.zero_grad()
            loss = ssl_wrapper(patches, node_t, adj_t)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f'SSL Epoch {epoch:02d}/{ssl_epochs:02d} | Reconstruction Loss: {total_loss / len(all_loader):.4f}')

    return base_model.state_dict()

pretrained_weights = pretrain_ssl(
    X_pca, segments, node_features, adj_norm, num_classes, ssl_epochs=10, batch_size=256
)

--- Starting Self-Supervised Pretraining on all 111104 pixels ---


/tmp/ipykernel_1128/100117869.py:55: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.vit_encoder = nn.TransformerEncoder(enc_layer, num_layers=2)


SSL Epoch 01/10 | Reconstruction Loss: 2.2714
SSL Epoch 02/10 | Reconstruction Loss: 1.2799
SSL Epoch 03/10 | Reconstruction Loss: 1.1950
SSL Epoch 04/10 | Reconstruction Loss: 1.1680
SSL Epoch 05/10 | Reconstruction Loss: 1.1718
SSL Epoch 06/10 | Reconstruction Loss: 1.1082
SSL Epoch 07/10 | Reconstruction Loss: 1.1086
SSL Epoch 08/10 | Reconstruction Loss: 1.0789
SSL Epoch 09/10 | Reconstruction Loss: 1.0708
SSL Epoch 10/10 | Reconstruction Loss: 1.0806


In [ ]:
# Cell 9: Plain ViT Baseline & Graph-Supervised Semi-Supervised Trainer
class PlainViTNet(nn.Module):
    """Plain ViT baseline (local branch only, no graph features or graph supervision)."""
    def __init__(self, in_channels=30, num_classes=16, embed_dim=128):
        super().__init__()
        self.vit_embed = PatchEmbedding(in_channels, embed_dim, patch_size=PATCH_SIZE)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=256,
            dropout=0.1, activation='gelu', batch_first=True, norm_first=True
        )
        self.vit_encoder = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, patches, seg_ids=None, node_feats=None, adj_norm=None):
        v = self.vit_embed(patches)
        v = self.vit_encoder(v).mean(dim=1)
        return self.classifier(v)


def compute_graph_supervision_losses(
    model, unlab_patches, unlab_segs, node_t, adj_t, num_classes,
    conf_thresh=0.88, num_nodes=None
):
    """
    1. Superpixel Consistency Loss: Pushes pixels within the same superpixel
       to predict consistent class probability distributions.
    2. Graph-Guided Pseudo-Label Propagation: Propagates confident superpixel
       distributions across adj_norm to neighbouring unlabelled pixels.
    """
    logits = model(unlab_patches, unlab_segs, node_t, adj_t)
    probs = F.softmax(logits, dim=-1)
    log_probs = F.log_softmax(logits, dim=-1)

    if num_nodes is None:
        num_nodes = node_t.size(0)

    sp_prob_sum = torch.zeros(num_nodes, num_classes, device=logits.device)
    sp_count = torch.zeros(num_nodes, 1, device=logits.device)
    sp_prob_sum.index_add_(0, unlab_segs, probs.detach())
    sp_count.index_add_(0, unlab_segs, torch.ones_like(unlab_segs, dtype=torch.float32).unsqueeze(-1))

    valid_sp = (sp_count.squeeze(-1) > 0)
    sp_mean_probs = sp_prob_sum / sp_count.clamp(min=1.0)

    # (A) Intra-superpixel consistency loss
    target_sp_probs = sp_mean_probs[unlab_segs].clamp(min=1e-6)
    loss_consistency = F.kl_div(log_probs, target_sp_probs, reduction='batchmean')

    # (B) Graph neighbor propagation of confident pseudo-labels
    propagated_sp_probs = torch.matmul(adj_t, sp_mean_probs)
    propagated_sp_probs = propagated_sp_probs / propagated_sp_probs.sum(dim=-1, keepdim=True).clamp(min=1e-6)

    pixel_prop_probs = propagated_sp_probs[unlab_segs]
    max_conf, pseudo_targets = pixel_prop_probs.max(dim=-1)
    conf_mask = (max_conf >= conf_thresh) & valid_sp[unlab_segs]

    if conf_mask.sum() > 0:
        loss_pseudo = F.cross_entropy(logits[conf_mask], pseudo_targets[conf_mask])
    else:
        loss_pseudo = torch.tensor(0.0, device=logits.device)

    return loss_consistency, loss_pseudo


def train_supervised_baseline(model, train_loader, test_loader, epochs=15, lr=1e-3):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    node_t = torch.tensor(node_features, dtype=torch.float32, device=device)
    adj_t = torch.tensor(adj_norm, dtype=torch.float32, device=device)

    for _ in range(epochs):
        model.train()
        for patches, labels, seg_ids in train_loader:
            patches, labels, seg_ids = patches.to(device), labels.to(device), seg_ids.to(device)
            optimizer.zero_grad()
            out = model(patches, seg_ids, node_t, adj_t)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()

    return evaluate(model, test_loader, node_t, adj_t)


def train_label_efficient_graph(
    pretrained_state, lab_loader, unlab_loader, test_loader,
    epochs=15, lr=8e-4, lambda_cons=0.5, lambda_pseudo=0.5
):
    model = ViTGraphNet(in_channels=X_pca.shape[2], num_classes=num_classes).to(device)
    model.load_state_dict(copy.deepcopy(pretrained_state))

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    node_t = torch.tensor(node_features, dtype=torch.float32, device=device)
    adj_t = torch.tensor(adj_norm, dtype=torch.float32, device=device)

    unlab_iter = iter(unlab_loader)

    for epoch in range(1, epochs + 1):
        model.train()
        ramp = min(1.0, epoch / 5.0)

        for patches_l, labels_l, segs_l in lab_loader:
            patches_l, labels_l, segs_l = patches_l.to(device), labels_l.to(device), segs_l.to(device)

            try:
                patches_u, segs_u = next(unlab_iter)
            except StopIteration:
                unlab_iter = iter(unlab_loader)
                patches_u, segs_u = next(unlab_iter)

            patches_u, segs_u = patches_u.to(device), segs_u.to(device)

            optimizer.zero_grad()
            logits_l = model(patches_l, segs_l, node_t, adj_t)
            loss_sup = criterion(logits_l, labels_l)

            loss_cons, loss_pseudo = compute_graph_supervision_losses(
                model, patches_u, segs_u, node_t, adj_t, num_classes
            )

            loss = loss_sup + ramp * (lambda_cons * loss_cons + lambda_pseudo * loss_pseudo)
            loss.backward()
            optimizer.step()

    return evaluate(model, test_loader, node_t, adj_t)

In [ ]:
# Cell 10: Run Benchmark Across 1%, 5%, 10% Label Fractions & Plot Accuracy Curves
label_fractions = [0.01, 0.05, 0.10]
results = {
    'Plain ViT': {'OA': [], 'AA': [], 'Kappa': []},
    'Exp 1: ViT + Graph (Sup only)': {'OA': [], 'AA': [], 'Kappa': []},
    'Ours: SSL + Graph Supervision': {'OA': [], 'AA': [], 'Kappa': []},
}

# Fixed 80% held-out test set so all label fractions are evaluated on identical test pixels
all_labelled_coords = [(r, c) for r in range(y.shape[0]) for c in range(y.shape[1]) if y[r, c] > 0]
all_labelled_targets = [y[r, c] - 1 for r, c in all_labelled_coords]

pool_coords, fixed_test_coords, pool_y, fixed_test_y = train_test_split(
    all_labelled_coords, all_labelled_targets,
    test_size=0.80, stratify=all_labelled_targets, random_state=SEED
)

fixed_test_ds = PatchDataset(X_pca, y, segments, fixed_test_coords, patch_size=PATCH_SIZE)
fixed_test_loader = DataLoader(fixed_test_ds, batch_size=256, shuffle=False)

unlab_ds = AllPixelsPatchDataset(X_pca, segments, patch_size=PATCH_SIZE)
unlab_loader = DataLoader(unlab_ds, batch_size=256, shuffle=True, drop_last=True)

for frac in label_fractions:
    sample_ratio = min(0.99, frac / 0.20)
    sub_coords, _, _, _ = train_test_split(
        pool_coords, pool_y, train_size=sample_ratio, stratify=pool_y, random_state=SEED
    )
    print(f'\n================ Label Fraction: {int(frac*100)}% ({len(sub_coords)} labelled pixels) ================')

    sub_ds = PatchDataset(X_pca, y, segments, sub_coords, patch_size=PATCH_SIZE)
    sub_loader = DataLoader(sub_ds, batch_size=min(64, len(sub_ds)), shuffle=True)

    # 1. Plain ViT
    vit_model = PlainViTNet(in_channels=PCA_COMPONENTS, num_classes=num_classes).to(device)
    oa, aa, kp = train_supervised_baseline(vit_model, sub_loader, fixed_test_loader, epochs=15)
    results['Plain ViT']['OA'].append(oa * 100)
    results['Plain ViT']['AA'].append(aa * 100)
    results['Plain ViT']['Kappa'].append(kp * 100)
    print(f'[Plain ViT]                    OA: {oa*100:.2f}% | AA: {aa*100:.2f}% | Kappa: {kp*100:.2f}%')

    # 2. Exp 1: Supervised ViT + Graph
    exp1_model = ViTGraphNet(in_channels=PCA_COMPONENTS, num_classes=num_classes).to(device)
    oa, aa, kp = train_supervised_baseline(exp1_model, sub_loader, fixed_test_loader, epochs=15)
    results['Exp 1: ViT + Graph (Sup only)']['OA'].append(oa * 100)
    results['Exp 1: ViT + Graph (Sup only)']['AA'].append(aa * 100)
    results['Exp 1: ViT + Graph (Sup only)']['Kappa'].append(kp * 100)
    print(f'[Exp 1: ViT+Graph (Sup only)]  OA: {oa*100:.2f}% | AA: {aa*100:.2f}% | Kappa: {kp*100:.2f}%')

    # 3. Proposed: SSL Pretraining + Superpixel Consistency + Graph Pseudo-Label Propagation
    oa, aa, kp = train_label_efficient_graph(
        pretrained_weights, sub_loader, unlab_loader, fixed_test_loader, epochs=15
    )
    results['Ours: SSL + Graph Supervision']['OA'].append(oa * 100)
    results['Ours: SSL + Graph Supervision']['AA'].append(aa * 100)
    results['Ours: SSL + Graph Supervision']['Kappa'].append(kp * 100)
    print(f'[Ours: SSL + Graph Sup]        OA: {oa*100:.2f}% | AA: {aa*100:.2f}% | Kappa: {kp*100:.2f}%')

# Plot Accuracy vs. Label Fraction
pct_labels = [int(f * 100) for f in label_fractions]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

styles = {
    'Plain ViT': ('o--', '#7f8c8d'),
    'Exp 1: ViT + Graph (Sup only)': ('s-', '#2980b9'),
    'Ours: SSL + Graph Supervision': ('^-', '#27ae60'),
}

for metric_idx, metric in enumerate(['OA', 'AA']):
    ax = axes[metric_idx]
    for name, (style, color) in styles.items():
        ax.plot(pct_labels, results[name][metric], style, color=color, linewidth=2.2, markersize=8, label=name)
    ax.set_title(f'{metric} vs. Label Fraction (Salinas)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Labelled Training Pixels (%)', fontsize=11)
    ax.set_ylabel(f'{metric} (%)', fontsize=11)
    ax.set_xticks(pct_labels)
    ax.set_xticklabels([f'{p}%' for p in pct_labels])
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('label_efficient_graph_learning.png', dpi=300)
plt.show()


================ Label Fraction: 1% (541 labelled pixels) ================


/tmp/ipykernel_1128/2424998802.py:11: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.vit_encoder = nn.TransformerEncoder(enc_layer, num_layers=2)


[Plain ViT]                    OA: 93.43% | AA: 96.25% | Kappa: 92.67%


/tmp/ipykernel_1128/100117869.py:55: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.vit_encoder = nn.TransformerEncoder(enc_layer, num_layers=2)


[Exp 1: ViT+Graph (Sup only)]  OA: 91.92% | AA: 95.19% | Kappa: 90.95%


/tmp/ipykernel_1128/100117869.py:55: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.vit_encoder = nn.TransformerEncoder(enc_layer, num_layers=2)


... (2 outputs left out)

[Plain ViT]                    OA: 97.22% | AA: 98.06% | Kappa: 96.90%


/tmp/ipykernel_1128/100117869.py:55: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.vit_encoder = nn.TransformerEncoder(enc_layer, num_layers=2)


[Exp 1: ViT+Graph (Sup only)]  OA: 97.57% | AA: 98.74% | Kappa: 97.29%


/tmp/ipykernel_1128/100117869.py:55: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.vit_encoder = nn.TransformerEncoder(enc_layer, num_layers=2)
